In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("blue-owls-assessment") \
    .config("spark.sql.warehouse.dir","spark-warehouse") \
    .enableHiveSupport() \
    .getOrCreate()


### Customers

In [3]:
customer= spark.read.option("header", "true").format("csv").load("work/output/silver/customers.csv")
customer = customer.filter("_is_valid=True").dropDuplicates(['customer_unique_id'])

orders = spark.read.option("header", "true").format("csv").load("work/output/silver/orders.csv")

customer_orders = customer.join(orders, on=['customer_id'])

spec = Window.partitionBy("customer_unique_id").orderBy(col("order_purchase_timestamp").desc())

customer_orders = customer_orders.withColumn("row_num", row_number().over(spec))

items = spark.read.option("header","true").option("inferSchema", "true").format("csv").load("work/output/silver/order_items.csv")
items = items.groupby("order_id").agg(sum(col("price") +col("freight_value")).alias("total_spend"))

customer_orders = customer_orders.join(items, on=['order_id'])

customer_orders = customer_orders.groupBy("customer_unique_id").agg(
    first(when(col("row_num") == 1, col("customer_city"))).alias("customer_city"),
    first(when(col("row_num") == 1, col("customer_state"))).alias("customer_state"),
    first(when(col("row_num") == 1, col("customer_zip_code_prefix"))).alias("customer_zip_code_prefix"),
    min(to_date("order_purchase_timestamp")).alias("first_order_date"),
    countDistinct(col("order_id")).alias("total_orders"),
    sum("total_spend").alias("total_spend")).withColumn('customer_key', monotonically_increasing_id()).withColumn("is_repeat_customer", when(col("total_orders") > 1, True).otherwise(False))

customer_orders.write.mode("overwrite").format("parquet").saveAsTable("dim_customers")

### Products

In [5]:
# 1. Load Silver Tables
products = spark.read.csv("work/output/silver/products.csv", header=True, inferSchema=True)

In [9]:
dim_products = (products.dropDuplicates(['product_id'])
.withColumn("product_key", monotonically_increasing_id())
.withColumn("product_category_name", when(col("product_category_name").isNull(), lit("UNKNOWN")).otherwise(col("product_category_name")))
.withColumn("product_volume_cm3",when(col("product_length_cm").isNull() | col("product_height_cm").isNull() | col("product_width_cm").isNull(), None)
.otherwise(col("product_width_cm") * col("product_height_cm") * col("product_length_cm"))).select("product_key", "product_id", "product_category_name", "product_weight_g", "product_volume_cm3", "product_photos_qty","product_description_lenght") 
)

In [10]:
dim_products.write.mode("overwrite").format("parquet").saveAsTable("dim_products")

### Sellers

In [11]:
sellers = spark.read.csv("work/output/silver/sellers.csv", header=True, inferSchema=True).dropDuplicates(['product_id'])

In [13]:
sellers = sellers.withColumn("seller_key",  F.monotonically_increasing_id()).select("seller_key", "seller_id"
, "seller_city", "seller_state", "seller_zip_code_prefix")                                                                  

sellers.write.mode("overwrite").format("parquet").saveAsTable("dim_sellers")                                                              

### Fact Order Items

In [15]:
# 1. Load Silver Tables
payments = spark.read.csv("work/output/silver/payments.csv", header=True, inferSchema=True)
items = spark.read.option("header","true").option("inferSchema", "true").format("csv").load("work/output/silver/order_items.csv")

In [18]:
# 2. Process Payments (Highest value type and Max installments per Order)
pay_window = Window.partitionBy("order_id").orderBy(desc("payment_value"), asc("payment_type"))

order_payments = payments.withColumn("rn", row_number().over(pay_window)) \
    .groupBy("order_id") \
    .agg(
        sum("payment_value").alias("total_order_payment"),
        first(when(col("rn") == 1, col("payment_type"))).alias("payment_type"),
        max("payment_installments").alias("payment_installments")
    )

In [19]:

order_totals = items.groupBy("order_id").agg(sum("price").alias("total_items_price"))

items_with_payment = items.join(order_totals, on="order_id") \
    .join(order_payments, on="order_id") \
    .withColumn("item_share", col("price") / col("total_items_price")) \
    .withColumn("distributed_payment", (col("item_share") * col("total_order_payment")).cast("decimal(18,2)"))


In [23]:
fact_order_items = items_with_payment.join(orders, on="order_id")

['order_id',
 'freight_value',
 'order_item_id',
 'price',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 '_ingested_at',
 '_source_endpoint',
 '_is_valid',
 'total_items_price',
 'total_order_payment',
 'payment_type',
 'payment_installments',
 'item_share',
 'distributed_payment',
 'customer_id',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'order_purchase_timestamp',
 'order_status',
 '_ingested_at',
 '_source_endpoint',
 '_is_valid']

In [32]:
customers_silver = spark.read.csv("work/output/silver/customers.csv", header=True, inferSchema=True) \
    .select("customer_id", "customer_unique_id")

fact_order_items = fact_order_items.join(customers_silver, on="customer_id", how="left")

fact_order_items = fact_order_items.join(dim_products.select("product_id", "product_key"), on='product_id', how='left').join(sellers.select("seller_id", "seller_key"), on='seller_id', how='left')

fact_order_items = fact_order_items.join(customer_orders, on="customer_unique_id", how="left") \
    .select(
        monotonically_increasing_id().alias("order_item_sk"),
        "order_id",
        "order_item_id",
        "customer_key", 
        "product_key",   
        "seller_key",    
        to_date("order_purchase_timestamp").alias("order_date"),
        "order_status",
        col("price").cast("decimal(18,2)"),
        col("freight_value").cast("decimal(18,2)"),
        col("distributed_payment").alias("payment_value"),
        "payment_type",
        "payment_installments",
        datediff("order_delivered_customer_date", "order_purchase_timestamp").alias("days_to_deliver"),
        datediff("order_delivered_customer_date", "order_estimated_delivery_date").alias("days_delivery_vs_estimate")
    ) \
    .withColumn("is_late_delivery", col("days_delivery_vs_estimate") > 0)

In [34]:
fact_order_items.write.mode("overwrite").format("parquet").saveAsTable("fact_order_items")